# Questions 4, 5, 6 — Theoretical Derivations

This notebook contains the mathematical derivations for the theoretical parts of the assignment, each followed by a small numerical verification cell.

---

# Question 4 — OLS, Ridge, and LASSO Estimators

Consider the simple **no-intercept** regression model:

$$y_i = \beta x_i + \epsilon_i, \quad i = 1, \ldots, n$$

where $\mathbb{E}[\epsilon_i \mid x] = 0$, $\mathrm{Var}(\epsilon_i \mid x) = \sigma^2$, and $\mathrm{Cov}(\epsilon_i, \epsilon_j) = 0$ for $i \neq j$.

## Q4.1 — OLS Estimator

### Derivation

Minimise the residual sum of squares:

$$\hat{\beta}_{\text{OLS}} = \arg\min_{\beta} \sum_{i=1}^n (y_i - \beta x_i)^2$$

Taking the derivative and setting it to zero:

$$\frac{\partial}{\partial\beta} \sum_i (y_i - \beta x_i)^2 = -2\sum_i x_i(y_i - \beta x_i) = 0$$

$$\Rightarrow \quad \hat{\beta}_{\text{OLS}} = \frac{\sum_{i=1}^n x_i y_i}{\sum_{i=1}^n x_i^2}$$

### Expected Value

$$\mathbb{E}[\hat{\beta}_{\text{OLS}}] = \mathbb{E}\left[\frac{\sum x_i y_i}{\sum x_i^2}\right] = \frac{\sum x_i \mathbb{E}[y_i]}{\sum x_i^2} = \frac{\sum x_i (\beta x_i)}{\sum x_i^2} = \beta$$

**The OLS estimator is unbiased.**

### Variance

$$\mathrm{Var}(\hat{\beta}_{\text{OLS}}) = \mathrm{Var}\left(\frac{\sum x_i y_i}{\sum x_i^2}\right) = \frac{\sum x_i^2 \cdot \sigma^2}{(\sum x_i^2)^2} = \frac{\sigma^2}{\sum_{i=1}^n x_i^2}$$

### Consistency

Write $\hat{\beta}_{\text{OLS}} = \beta + \dfrac{\sum x_i \epsilon_i}{\sum x_i^2}$.

By the Weak Law of Large Numbers (WLLN), $\dfrac{1}{n}\sum x_i \epsilon_i \xrightarrow{p} \mathbb{E}[x_i \epsilon_i] = 0$ (since $\mathbb{E}[\epsilon_i \mid x] = 0$) and $\dfrac{1}{n}\sum x_i^2 \xrightarrow{p} \mathbb{E}[x_i^2] > 0$.

Therefore $\hat{\beta}_{\text{OLS}} - \beta = \dfrac{\frac{1}{n}\sum x_i \epsilon_i}{\frac{1}{n}\sum x_i^2} \xrightarrow{p} 0$,  so $\hat{\beta}_{\text{OLS}} \xrightarrow{p} \beta$. **The OLS estimator is consistent.**


## Q4.2 — Ridge Estimator ($\beta^2 \leq r$)

### Derivation

Ridge regression minimises the Lagrangian form:

$$\hat{\beta}_{\text{Ridge}} = \arg\min_{\beta} \sum_i (y_i - \beta x_i)^2 + \lambda \beta^2$$

The constraint $\beta^2 \leq r$ corresponds to a specific $\lambda \geq 0$. Taking the derivative:

$$-2\sum_i x_i(y_i - \beta x_i) + 2\lambda\beta = 0$$

$$\beta\left(\sum_i x_i^2 + \lambda\right) = \sum_i x_i y_i$$

$$\Rightarrow \quad \hat{\beta}_{\text{Ridge}} = \frac{\sum_{i=1}^n x_i y_i}{\sum_{i=1}^n x_i^2 + \lambda}$$

### Expected Value (Bias)

$$\mathbb{E}[\hat{\beta}_{\text{Ridge}}] = \frac{\sum x_i \cdot \beta x_i}{\sum x_i^2 + \lambda} = \frac{\beta \sum x_i^2}{\sum x_i^2 + \lambda} = \beta \cdot \frac{\sum x_i^2}{\sum x_i^2 + \lambda} \neq \beta$$

**Bias** $= \mathbb{E}[\hat{\beta}_{\text{Ridge}}] - \beta = -\dfrac{\lambda\beta}{\sum x_i^2 + \lambda}$ (shrinkage toward zero).

### Variance

$$\mathrm{Var}(\hat{\beta}_{\text{Ridge}}) = \frac{\sigma^2 \sum x_i^2}{(\sum x_i^2 + \lambda)^2}$$

This is **smaller** than the OLS variance (Ridge trades bias for variance reduction).

### Consistency

Write $S_{xx} = \sum x_i^2$. If $\lambda = \lambda(n)$ such that $\lambda/n \to 0$ as $n \to \infty$:

$$\hat{\beta}_{\text{Ridge}} = \frac{S_{xx}/n \cdot \hat{\beta}_{\text{OLS}}}{S_{xx}/n + \lambda/n} \xrightarrow{p} \frac{\mathbb{E}[x^2] \cdot \beta}{\mathbb{E}[x^2] + 0} = \beta$$

**Consistent** provided $\lambda/n \to 0$.


## Q4.3 — LASSO Estimator ($|\beta| \leq r$)

### Derivation

LASSO minimises:

$$\hat{\beta}_{\text{LASSO}} = \arg\min_{\beta} \sum_i (y_i - \beta x_i)^2 + \lambda |\beta|$$

The L1 penalty is non-differentiable at $\beta = 0$. Using the **subgradient condition**:

$$-2\sum_i x_i(y_i - \beta x_i) + \lambda \cdot \text{sign}(\beta) = 0 \quad (\text{if } \beta \neq 0)$$

Let $z = \sum x_i y_i / \sum x_i^2 = \hat{\beta}_{\text{OLS}}$ and $S_{xx} = \sum x_i^2$. Then:

$$\hat{\beta}_{\text{LASSO}} = \text{sign}(z) \cdot \max\left(|z| - \frac{\lambda}{2 S_{xx}}, \; 0\right)$$

This is the **soft-thresholding** operator: coefficients smaller than $\lambda / (2 S_{xx})$ in absolute value are set exactly to zero (hard sparsity — unlike Ridge).

### Expected Value (Bias)

The bias depends on whether the true $\beta$ is in the zero region or not.  
Let $\tau = \lambda / (2 S_{xx})$:

$$\hat{\beta}_{\text{LASSO}} = \text{sign}(z) (|z| - \tau)^+$$

For $|\beta| > \tau$ (survivor case): $\mathbb{E}[\hat{\beta}_{\text{LASSO}}] \approx \beta - \tau \cdot \text{sign}(\beta)$ (downward bias in magnitude).  
For $|\beta| \leq \tau$ (zero case): $\mathbb{E}[\hat{\beta}_{\text{LASSO}}] = 0$, bias $= -\beta$.

### Consistency

If $\lambda/n \to 0$, then $\tau = \lambda / (2 S_{xx}) = (\lambda/n) / (2 S_{xx}/n) \to 0$.

Since $\hat{\beta}_{\text{OLS}} \xrightarrow{p} \beta$ and the soft-threshold $\tau \to 0$, we have:

$$\hat{\beta}_{\text{LASSO}} = \text{sign}(z)(|z| - \tau)^+ \xrightarrow{p} \beta \quad \text{(provided } \beta \neq 0 \text{ or } \tau \to 0)$$

**Consistent** provided $\lambda/n \to 0$.


In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, Lasso

# ── Monte Carlo verification of Q4 ──────────────────────────────────────
# True model: y = beta * x + eps,  beta = 1.5
true_beta = 1.5
sigma     = 1.0
n_reps    = 2000
lam       = 0.5  # fixed penalty (for demonstration)

results = []
for n in [50, 500, 5000]:
    ols_estimates   = []
    ridge_estimates = []
    lasso_estimates = []

    for _ in range(n_reps):
        x   = np.random.randn(n)
        eps = np.random.randn(n) * sigma
        y   = true_beta * x + eps

        # OLS (closed form)
        beta_ols = np.dot(x, y) / np.dot(x, x)
        ols_estimates.append(beta_ols)

        # Ridge (closed form)
        Sxx = np.dot(x, x)
        beta_ridge = np.dot(x, y) / (Sxx + lam)
        ridge_estimates.append(beta_ridge)

        # LASSO (soft-threshold)
        z   = np.dot(x, y) / Sxx
        tau = lam / (2 * Sxx)
        beta_lasso = np.sign(z) * max(abs(z) - tau, 0)
        lasso_estimates.append(beta_lasso)

    results.append({
        'n':             n,
        'OLS_bias':      round(np.mean(ols_estimates) - true_beta, 5),
        'OLS_var':       round(np.var(ols_estimates), 5),
        'Ridge_bias':    round(np.mean(ridge_estimates) - true_beta, 5),
        'Ridge_var':     round(np.var(ridge_estimates), 5),
        'LASSO_bias':    round(np.mean(lasso_estimates) - true_beta, 5),
        'LASSO_var':     round(np.var(lasso_estimates), 5),
    })

mc_df = pd.DataFrame(results)
print(f'True beta = {true_beta},  lambda = {lam},  n_reps = {n_reps}')
print('\nMonte Carlo bias and variance:')
display(mc_df)
print('\nObs: OLS bias -> 0 for all n (unbiased).')
print('Ridge bias shrinks as n grows (lambda/n -> 0 => consistent).')
print('LASSO bias shrinks as n grows (tau -> 0 => consistent).')
print('Both Ridge and LASSO have lower variance than OLS at small n (bias-variance tradeoff).')


True beta = 1.5,  lambda = 0.5,  n_reps = 2000

Monte Carlo bias and variance:


,n,OLS_bias,OLS_var,Ridge_bias,Ridge_var,LASSO_bias,LASSO_var
0,50,0.00014,0.02024,-0.01530,0.01978,-0.00506,0.02023
1,500,0.00083,0.00197,-0.00067,0.00197,0.00033,0.00197
2,5000,-0.00045,0.00021,-0.00060,0.00021,-0.00050,0.00021



Obs: OLS bias -> 0 for all n (unbiased).
Ridge bias shrinks as n grows (lambda/n -> 0 => consistent).
LASSO bias shrinks as n grows (tau -> 0 => consistent).
Both Ridge and LASSO have lower variance than OLS at small n (bias-variance tradeoff).


---
# Question 5 — AIC for the Gaussian Linear Model

**Claim:** For the model $y_i = \beta_0 + \beta_1 x_{1i} + \cdots + \beta_p x_{pi} + \epsilon_i$ with $\epsilon_i \overset{iid}{\sim} N(0,\sigma^2)$, the AIC reduces to:

$$AIC = n \log(\hat{\sigma}^2_{\text{MLE}}) + C_{n,p}$$

where $\hat{\sigma}^2_{\text{MLE}} = RSS/n$ and $C_{n,p}$ depends only on $n$ and $p$.

## Derivation

### Step 1 — Log-likelihood of the Gaussian linear model

With $\epsilon_i \overset{iid}{\sim} N(0,\sigma^2)$, each observation has density $f(y_i) = \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left(-\frac{(y_i - \mathbf{x}_i^\top\boldsymbol{\beta})^2}{2\sigma^2}\right)$.

The log-likelihood is:

$$\ell(\boldsymbol{\beta}, \sigma^2) = -\frac{n}{2}\log(2\pi) - \frac{n}{2}\log(\sigma^2) - \frac{1}{2\sigma^2} \sum_{i=1}^n (y_i - \mathbf{x}_i^\top\boldsymbol{\beta})^2$$

$$= -\frac{n}{2}\log(2\pi) - \frac{n}{2}\log(\sigma^2) - \frac{RSS(\boldsymbol{\beta})}{2\sigma^2}$$

### Step 2 — Plug in the MLEs

The MLE of $\boldsymbol{\beta}$ is the OLS estimator: $\hat{\boldsymbol{\beta}}_{\text{MLE}} = (\mathbf{X}^\top\mathbf{X})^{-1}\mathbf{X}^\top\mathbf{y}$,  
and the MLE of $\sigma^2$ is $\hat{\sigma}^2_{\text{MLE}} = RSS/n$ (note: biased, divides by $n$ not $n-p-1$).

Substituting into the maximised log-likelihood:

$$\ell(\hat{\boldsymbol{\beta}}, \hat{\sigma}^2_{\text{MLE}}) = -\frac{n}{2}\log(2\pi) - \frac{n}{2}\log(\hat{\sigma}^2_{\text{MLE}}) - \frac{RSS}{2 \cdot (RSS/n)} = -\frac{n}{2}\log(2\pi) - \frac{n}{2}\log(\hat{\sigma}^2_{\text{MLE}}) - \frac{n}{2}$$

### Step 3 — Compute AIC

The number of free parameters is $p + 1$ (the $p+1$ regression coefficients) **plus** $1$ for $\sigma^2$, giving $p + 2$ total parameters. AIC penalises by $2 \times (p+2)$:

$$AIC = -2\ell(\hat{\boldsymbol{\beta}}, \hat{\sigma}^2) + 2(p + 2)$$

$$= -2\left[-\frac{n}{2}\log(2\pi) - \frac{n}{2}\log(\hat{\sigma}^2_{\text{MLE}}) - \frac{n}{2}\right] + 2(p+2)$$

$$= n\log(2\pi) + n\log(\hat{\sigma}^2_{\text{MLE}}) + n + 2(p+2)$$

$$\boxed{AIC = n\log(\hat{\sigma}^2_{\text{MLE}}) + C_{n,p}}$$

where $C_{n,p} = n\log(2\pi) + n + 2(p+2)$ depends only on $n$ and $p$. $\square$

**Implication:** When comparing models with the same $n$, minimising AIC is equivalent to minimising $n\log(\hat{\sigma}^2_{\text{MLE}})$, i.e., the log of the MLE residual variance (plus a complexity penalty through $p$ in $C_{n,p}$).


In [2]:
import numpy as np
import statsmodels.api as sm

# ── Numerical verification of Q5 ────────────────────────────────────────
np.random.seed(0)
n = 100
X_sim = np.column_stack([np.ones(n),
                          np.random.randn(n),
                          np.random.randn(n)])
y_sim = 1.0 + 2.0 * X_sim[:, 1] - 0.5 * X_sim[:, 2] + np.random.randn(n)

ols_sim = sm.OLS(y_sim, X_sim).fit()

# statsmodels AIC
aic_statsmodels = ols_sim.aic

# Our closed-form: n * log(sigma_hat_MLE^2) + C_{n,p}
# p = 2 predictors (excluding intercept), total params = p+2 = 4
p_params      = 2          # number of predictors (not counting intercept)
RSS           = ols_sim.ssr
sigma2_mle    = RSS / n    # MLE: divide by n
C_np          = n * np.log(2 * np.pi) + n + 2 * (p_params + 2)
aic_closed    = n * np.log(sigma2_mle) + C_np

print(f'AIC from statsmodels   : {aic_statsmodels:.6f}')
print(f'AIC from closed form   : {aic_closed:.6f}')
diff = aic_closed - aic_statsmodels
print(f'Difference (closed - sm): {diff:.4f}')
print()
print('The difference is exactly 2.0.')
print('This is a well-known parameter-counting convention difference:')
print('  Our derivation: k = (p+1) betas + 1 sigma = p+2 free parameters')
print('  statsmodels OLS: k = (p+1) betas only (sigma is concentrated out')
print('    of the profile likelihood, not counted as a free parameter) = p+1')
print('  Difference in penalty: 2*(p+2) - 2*(p+1) = 2  <-- exactly what we see')
print()
print('Important: this constant offset DOES NOT affect model selection.')
print('Comparing model A vs model B, both AICs shift by the same +2, so')
print('the relative ranking (which model wins) is identical.')
print('For comparisons, always use model.aic from statsmodels directly.')


AIC from statsmodels   : 279.138791
AIC from closed form   : 281.138791
Difference (closed - sm): 2.0000

The difference is exactly 2.0.
This is a well-known parameter-counting convention difference:
  Our derivation: k = (p+1) betas + 1 sigma = p+2 free parameters
  statsmodels OLS: k = (p+1) betas only (sigma is concentrated out
    of the profile likelihood, not counted as a free parameter) = p+1
  Difference in penalty: 2*(p+2) - 2*(p+1) = 2  <-- exactly what we see

Important: this constant offset DOES NOT affect model selection.
Comparing model A vs model B, both AICs shift by the same +2, so
the relative ranking (which model wins) is identical.
For comparisons, always use model.aic from statsmodels directly.


---
# Question 6 — Discrete Uniform Distribution and Rank Sums

## Q6a — Expected Value and Variance of $X \sim DU[1, N]$

### Expected Value

$$\mathbb{E}[X] = \frac{1}{N} \sum_{k=1}^N k = \frac{1}{N} \cdot \frac{N(N+1)}{2} = \frac{N+1}{2}$$

### Variance

We use $\mathrm{Var}(X) = \mathbb{E}[X^2] - (\mathbb{E}[X])^2$.

$$\mathbb{E}[X^2] = \frac{1}{N}\sum_{k=1}^N k^2 = \frac{1}{N} \cdot \frac{N(N+1)(2N+1)}{6} = \frac{(N+1)(2N+1)}{6}$$

$$\mathrm{Var}(X) = \frac{(N+1)(2N+1)}{6} - \left(\frac{N+1}{2}\right)^2 = \frac{(N+1)}{1}\left[\frac{2N+1}{6} - \frac{N+1}{4}\right]$$

$$= (N+1)\cdot\frac{2(2N+1) - 3(N+1)}{12} = (N+1)\cdot\frac{4N+2 - 3N - 3}{12} = (N+1)\cdot\frac{N-1}{12} = \frac{N^2 - 1}{12}$$

$$\boxed{\mathbb{E}[X] = \frac{N+1}{2}, \qquad \mathrm{Var}(X) = \frac{N^2-1}{12}}$$


## Q6b — Rank Sums $R_1$ and $R_2$

We have two groups of sizes $n_1$ and $n_2$ with $n_1 + n_2 = N$. The combined sample is ranked $1$ through $N$. $R_1$ and $R_2$ are the sum of ranks in each group.

### Q6b.1 — $R_1 + R_2$ in terms of $N$

Since every integer from $1$ to $N$ appears exactly once across both groups:

$$R_1 + R_2 = \sum_{k=1}^N k = \frac{N(N+1)}{2}$$

### Q6b.2 — $\mathbb{E}[R_1]$ and $\mathbb{E}[R_2]$ under $H_0$

Under the null hypothesis of **no difference in group distributions**, all $\binom{N}{n_1}$ rank assignments are equally likely. Each of the $N$ ranks is equally likely to be assigned to any observation. For group 1 containing $n_1$ observations, each rank has probability $n_1/N$ of belonging to group 1:

$$\mathbb{E}[R_1] = \sum_{k=1}^N k \cdot P(\text{rank } k \in \text{group 1}) = \frac{n_1}{N} \sum_{k=1}^N k = \frac{n_1}{N} \cdot \frac{N(N+1)}{2} = \frac{n_1(N+1)}{2}$$

Similarly:

$$\mathbb{E}[R_2] = \frac{n_2(N+1)}{2}$$

**Cross-check:** $\mathbb{E}[R_1] + \mathbb{E}[R_2] = \frac{(n_1+n_2)(N+1)}{2} = \frac{N(N+1)}{2}$ ✓

These are the null expected rank sums used in the **Wilcoxon rank-sum test** (also known as the Mann-Whitney U test). A large deviation of $R_1$ from $n_1(N+1)/2$ provides evidence against $H_0$.


In [3]:
import numpy as np

# ── Verify Q6 analytically and by simulation ────────────────────────────

# Q6a: Discrete Uniform moments
N = 20
samples_du = np.arange(1, N + 1)

empirical_mean = np.mean(samples_du)
analytical_mean = (N + 1) / 2

empirical_var  = np.var(samples_du, ddof=0)
analytical_var = (N**2 - 1) / 12

print('Q6a — Discrete Uniform DU[1, N], N =', N)
print(f'  E[X]:   analytical = {analytical_mean:.4f},  empirical = {empirical_mean:.4f}')
print(f'  Var(X): analytical = {analytical_var:.4f},  empirical = {empirical_var:.4f}')

# Q6b: Rank sum expected values by simulation
np.random.seed(123)
n1, n2  = 15, 20
N_total = n1 + n2
n_sims  = 50_000

R1_sims = np.empty(n_sims)
for i in range(n_sims):
    # Randomly assign ranks 1..N to two groups (uniform assignment = H0)
    ranks = np.random.permutation(N_total) + 1  # ranks 1..N
    R1_sims[i] = ranks[:n1].sum()

analytical_E_R1 = n1 * (N_total + 1) / 2
analytical_E_R2 = n2 * (N_total + 1) / 2

print(f'\nQ6b — Rank sums under H0, n1={n1}, n2={n2}, N={N_total}')
print(f'  E[R1]: analytical = {analytical_E_R1:.2f},  simulation = {R1_sims.mean():.2f}')
print(f'  E[R2]: analytical = {analytical_E_R2:.2f},  simulation = {N_total*(N_total+1)/2 - R1_sims.mean():.2f}')
print(f'  R1+R2 = N(N+1)/2 = {N_total*(N_total+1)//2}  (always true by construction)')


Q6a — Discrete Uniform DU[1, N], N = 20
  E[X]:   analytical = 10.5000,  empirical = 10.5000
  Var(X): analytical = 33.2500,  empirical = 33.2500



Q6b — Rank sums under H0, n1=15, n2=20, N=35
  E[R1]: analytical = 270.00,  simulation = 269.97
  E[R2]: analytical = 360.00,  simulation = 360.03
  R1+R2 = N(N+1)/2 = 630  (always true by construction)
